In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

In [2]:
train = pd.read_csv("Train_dataset.csv")
test = pd.read_csv("Test_dataset.csv")

print(train.shape)
print(test.shape)

(1966, 9)
(312, 8)


In [3]:
# Remove missing target
train = train.dropna(subset=["age_group"]).reset_index(drop=True)

# Encode target
train["age_group"] = train["age_group"].map({
    "Adult":0,
    "Senior":1
})

# Drop identifier
train.drop(columns=["SEQN"], inplace=True)
test.drop(columns=["SEQN"], inplace=True)

# Fill missing values

for col in train.columns:

    if col=="age_group":
        continue

    median=train[col].median()

    train[col]=train[col].fillna(median)

    test[col]=test[col].fillna(median)

print(train.isnull().sum())

RIAGENDR     0
PAQ605       0
BMXBMI       0
LBXGLU       0
DIQ010       0
LBXGLT       0
LBXIN        0
age_group    0
dtype: int64


In [14]:
# ==========================================
# CELL 4 : Final Medical Feature Engineering
# ==========================================

def feature_engineering(df):

    # BMI Category
    df["BMI_Category"] = pd.cut(
        df["BMXBMI"],
        bins=[0, 18.5, 25, 30, 100],
        labels=[0, 1, 2, 3]
    ).astype(int)

    # Binary Features
    df["High_Glucose"] = (df["LBXGLU"] > 100).astype(int)
    df["High_Insulin"] = (df["LBXIN"] > 15).astype(int)

    # Interaction Features
    df["BMI_Glucose"] = df["BMXBMI"] * df["LBXGLU"]
    df["Diabetes_Glucose"] = df["DIQ010"] * df["LBXGLU"]
    df["Glucose_Insulin"] = df["LBXGLU"] * df["LBXIN"]

    df["Glucose_BMI_Ratio"] = df["LBXGLU"] / (df["BMXBMI"] + 1)
    df["Insulin_BMI_Ratio"] = df["LBXIN"] / (df["BMXBMI"] + 1)

    df["Glucose_GLT"] = df["LBXGLU"] * df["LBXGLT"]

    # Log Features
    df["Log_Insulin"] = np.log1p(df["LBXIN"])
    df["Log_GLT"] = np.log1p(df["LBXGLT"])

    # Polynomial Features
    df["BMI_Squared"] = df["BMXBMI"] ** 2
    df["Glucose_Squared"] = df["LBXGLU"] ** 2

    # ==========================
    # Clinical Features
    # ==========================

    # HOMA-IR
    df["HOMA_IR"] = (df["LBXGLU"] * df["LBXIN"]) / 405

    # QUICKI
    df["QUICKI"] = 1 / (
        np.log(df["LBXGLU"] + 1) +
        np.log(df["LBXIN"] + 1)
    )

    # OGTT Features
    df["OGTT_GLU_Ratio"] = df["LBXGLT"] / (df["LBXGLU"] + 1)
    df["OGTT_GLU_Diff"] = df["LBXGLT"] - df["LBXGLU"]
    df["OGTT_Insulin"] = df["LBXGLT"] * df["LBXIN"]

    # ==========================
    # NEW DOMAIN FEATURES
    # ==========================

    df["OGTT_Response"] = df["LBXGLT"] - df["LBXGLU"]

    df["OGTT_Fold"] = df["LBXGLT"] / (df["LBXGLU"] + 1)

    df["OGTT_BMI"] = (
        (df["LBXGLT"] - df["LBXGLU"]) /
        (df["BMXBMI"] + 1)
    )

    df["OGTT_Insulin_Ratio"] = (
        (df["LBXGLT"] - df["LBXGLU"]) /
        (df["LBXIN"] + 1)
    )

    df["Glucose_Clearance"] = (
        df["LBXGLU"] /
        (df["LBXGLT"] + 1)
    )

    df["Metabolic_Index"] = (
        df["HOMA_IR"] *
        df["OGTT_Fold"]
    )

    return df


train = feature_engineering(train)
test = feature_engineering(test)

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)
print("Total Features:", train.shape[1])

Train Shape: (1952, 32)
Test Shape : (312, 31)
Total Features: 32


In [15]:
# ==========================================
# CELL 5 : Prepare Features
# ==========================================

X = train.drop(columns=["age_group"])
y = train["age_group"]

print(X.shape)
print(y.shape)

(1952, 31)
(1952,)


In [16]:
# ==========================================
# CELL 6 : 5-Fold XGBoost
# ==========================================

from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

test_prob = np.zeros(len(test))

cv_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), 1):

    print(f"\n========== Fold {fold} ==========")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = XGBClassifier(
        subsample=0.9,
        scale_pos_weight=4,
        n_estimators=300,
        min_child_weight=5,
        max_depth=3,
        learning_rate=0.01,
        gamma=0.25,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)

    valid_prob = model.predict_proba(X_valid)[:,1]

    pred = (valid_prob >= 0.35).astype(int)

    score = f1_score(y_valid, pred)

    cv_scores.append(score)

    print("Fold F1:", score)

    test_prob += model.predict_proba(test)[:,1] / 5

print("\n==============================")
print("Average CV F1:", np.mean(cv_scores))
print("==============================")


========== Fold 1 ==========
Fold F1: 0.35856573705179284

========== Fold 2 ==========
Fold F1: 0.4473684210526316

========== Fold 3 ==========
Fold F1: 0.4444444444444444

========== Fold 4 ==========
Fold F1: 0.38222222222222224

========== Fold 5 ==========
Fold F1: 0.43902439024390244

Average CV F1: 0.41432504300299866


In [8]:
# ==========================================
# CELL 7 : Save Multiple Submissions
# ==========================================

for t in [0.30, 0.35, 0.40]:

    submission = pd.DataFrame({
        "age_group": (test_prob >= t).astype(int)
    })

    filename = f"submission_cv_{int(t*100)}.csv"

    submission.to_csv(filename, index=False)

    print(filename, "saved")

submission_cv_30.csv saved
submission_cv_35.csv saved
submission_cv_40.csv saved
